# Voca download and analysis

[Voca](https://voca.arizona.edu/) is the University of Arizona's audiovisual archive, storing a subset of fully transcribed recordings of the readings at the Poetry Center.

Due to ADA regulations, it turns out that we are good to do some web scraping to extract the transcripts and run some computational analysis.

The site uses an open-source content management system called [Drupal](https://new.drupal.org/). It will be migrated soon in a UI redesign project in which case Sarah Kortemeier can help get data (or help us understand how to get data).

## Install and load dependencies

We will be using a web scaping Python library called [Beautiful Soup](https://beautiful-soup-4.readthedocs.io/en/latest/).

In [ ]:
# run this code to make sure we have all of our dependencies installed
!pip install beautifulsoup4
!pip install requests

Now we need to import dependencies to tell Python we will be using them in our program.

In [ ]:
from bs4 import BeautifulSoup
import requests

## Get the Browse Reading by Year content

It contains the following about each poem:
- Reading title (with reading date)
- Reading description (with anthology publication names and dates)
- List of tags

First, we'll define a function to make get the readings on the HTML page, since we'll have multiple pages of poems to consider.

In [ ]:
# Use the browse by year pages to get a list of all readings (not poems yet!) in Voca
BROWSE_BY_YEAR_BASE_URL = "https://voca.arizona.edu/browse-by-year"

def get_readings_for_page(page_num, print_soup=False):
  """
  Returns the readings as iterable HTML objects in a page.
  Args:
    page_num (int): The page we want to try parsing.
    print_soup (bool): Optional flag to print the HTML for inspection.
  """
  # 1. Assemble the full url we want HTML for
  browse_by_year_page_url = f"{BROWSE_BY_YEAR_BASE_URL}?page={page_num}"

  # 2. Send the request and wait for the response!
  response = requests.get(browse_by_year_page_url)

  # 3. Turn the HTML (in response.txt) into a parsable soup object
  soup = BeautifulSoup(response.text, 'html.parser')

  # [Optional] Print out what the HTML looks like
  if print_soup:
    print(soup.prettify(formatter="html"))

  # 4. Get all row HTML elements
  return soup.find_all("div", {"class": 'views-row'})

In [ ]:
# print out the HTML to inspect elements

test_page_num = 24 # start on the first page
get_readings_for_page(test_page_num, True)

In [ ]:
# Let's get all the readings!
# We aren't sure how many pages there are, so we'll keep reading pages until the page we retreive has no readings.

current_page_num = 0
current_page_readings = get_readings_for_page(current_page_num)
all_page_readings_titles = []
all_page_readings_urls = []
all_page_readings_tags = []
all_page_readings_descriptions = []

while current_page_readings:
  print(f"Processing page {current_page_num}")

  for row_soup in current_page_readings:
    all_page_readings_titles.append(row_soup.find("div", {"class": "views-field views-field-title"}).text)
    all_page_readings_urls.append(row_soup.find("div", {"class": "views-field views-field-title"}).find("a").get("href"))
    all_page_readings_descriptions.append(row_soup.find("div", {"class": "views-field views-field-field-reading-description"}).text)

    tags_soup = row_soup.find("div", {"class": "views-field views-field-field-tags"})

    tags = []
    if tags_soup:
      for tag_soup in tags_soup.find_all("a"):
        tags.append(tag_soup.text)

    all_page_readings_tags.append(tags)

  current_page_num += 1
  current_page_readings = get_readings_for_page(current_page_num)



In [ ]:
all_page_readings_titles

## Put everything into Pandas for data inspection

Sanity check that data looks reasonable before we do things with it.

In [ ]:
import pandas as pd

readings_data = {
  'title': all_page_readings_titles,
  'url': all_page_readings_urls,
  'tags': all_page_readings_tags,
  'description': all_page_readings_descriptions
}

readings_df = pd.DataFrame(readings_data)

In [ ]:
readings_df

The dates are embedded in the titles, so let's split that out for our convenience.

In [ ]:
reading_dates_strings = readings_df["title"].str.split(' - ', expand=True).iloc[:,1].str.strip()
readings_df["reading_date"] = pd.to_datetime(reading_dates_strings, format="%B %d, %Y")

In [ ]:
readings_df

## Generate some basic visualizations with Matplotlib

In [ ]:
import matplotlib.pyplot as plt

readings_df['reading_date'].dt.to_period('Y').value_counts().sort_index().plot(kind='bar', figsize=(10, 5))

plt.title('Readings over the years in Voca')
plt.xlabel('Year')
plt.ylabel('Count')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

readings_df['reading_date'].dt.month.value_counts().sort_index().plot(kind='bar', figsize=(10, 5))

plt.title('Readings by the months in Voca')
plt.xlabel('Month')
plt.ylabel('Count')
plt.show()

In [ ]:
# save df as csv
readings_df.to_csv('readings_parsed.csv', index=False)

In [ ]:
# get tag frequency
all_tags = [tag for tags in all_page_readings_tags for tag in tags]
tag_counts = pd.Series(all_tags).value_counts()

In [ ]:
tag_counts

In [ ]:
# save that as a csv too
tag_counts.to_csv('tag_counts.csv')

# Get poems per reading

Every poetry reading has multiple poems. It comes with publication, poem title, and timing information.

The goal is to get this format:

```{json}
[{
  reading_title: "",
  reading_link: "",
  reading_date: "",
  reading_description: "",
  readers: [{reader_name: "", reader_link: ""}],
  reading_sponsors: "",
  reading_series: "",
  location: "",
  reading_tags: "",
  tracks: [
    {
      track_name: "",
      track_link: "", // in track-link class
      reader_name: "",
      reader_link: "",
      audio_file_link: "",
      captions_link: "",
      description: "",
      citation: "", // could be missing if not a poem
      tags: ""
    }, ...

  ]
}, ...
]
```

In [ ]:
# Genuinely helpful retreival function
def get_field_item_if_exists(field_soup):
  if not field_soup:
    return ""
  return field_soup.find("div", {"class": "field__item"}).text.strip()

Here are some functions to handle the top-level metadata that focuses on the overall reading.

In [ ]:
def get_readers_for_reading(soup):
  """
    <div class="field field--name-field-reader field--type-entity-reference field--label-above">
    <div class="field__label">Reader</div>
    <div class="field__items">
    <div class="field__item"><a href="/reader/last-first" hreflang="en">Last, First</a></div>
    </div>
  """
  readers_soup = section_soup.find("div", {"class": "field--name-field-reader"})
  readers = []
  if readers_soup:
    for reader_soup in readers_soup.find_all("a"):
      readers.append({
        "reader_name": reader_soup.text,
        "reader_link": reader_soup.get("href")
      })


def add_metadata_for_reading(soup, metadata):
  get_readers_for_reading(soup)
  sponsors_soup = section_soup.find("div", {"class": "field--name-field-reading-sponsors"})
  reading_series_soup = section_soup.find("div", {"class": "field--name-field-reading-series"})
  location_soup = section_soup.find("div", {"class": "field--name-field-reading-location"})
  tags_soup = section_soup.find("div", {"class": "field--name-field-tags"})

  # these attributes might be missing
  metadata['reading_sponsors'] = get_field_item_if_exists(sponsors_soup)
  metadata['reading_series'] = get_field_item_if_exists(reading_series_soup)
  metadata['location'] = get_field_item_if_exists(location_soup)

  # There could be multiple tags.
  metadata['reading_tags'] = []
  if tags_soup:
    for tag_soup in tags_soup.find_all("a"):
      metadata['reading_tags'].append(tag_soup.text)


Here are some functions that handle the second-layer metadata, focused on the segmented parts of the reading (e.g., Welcome, Introduction, Poem, etc).

In [ ]:
def resolve_audio_file(track_soup, track_metadata):
    # in 2000, beginning with Dana Levin's reading, the audio files started using mp4 (video instead of audio file)
    audio_soup = track_soup.find("div", {"class": "field--name-field-media-az-audio-file"})
    if not audio_soup:
      # check for video
      audio_soup = track_soup.find("div", {"class": "field--name-field-media-az-video-file"})

    track_metadata['audio_file_link'] = audio_soup.find("a").get("href")


Now this code makes all the API calls. It takes around 5-6 minutes to run.

In [ ]:
VOCA_BASE_URL = "https://voca.arizona.edu"

# open readings_parsed.csv
readings_df = pd.read_csv('readings_parsed.csv')

reading_details_list = []

# for each row (representing one reading), get 'url' and query for data
for index, row in readings_df.iterrows():

  reading_url = row['url']
  print(f"Processing {row['title']}")

  response = requests.get(VOCA_BASE_URL + reading_url)
  soup = BeautifulSoup(response.text, 'html.parser')

  # start to build metadata obj
  metadata = {
    "reading_title": row['title'],
    "reading_url": reading_url,
    "reading_date": row['reading_date'],
    "reading_description": row['description']
  }


  section_soup = soup.find("section", {"class": "section"}) # <section class="section">
  add_metadata_for_reading(section_soup, metadata)

  # iterate through each track in list of tracks
  tracks = []

  tracks_soup = section_soup.find("div", {"class": "field--name-field-reading-tracks"})

  # May Swenson and other readings may have a placeholder (only available at Poetry Center)
  if not tracks_soup:
    metadata['tracks'] = tracks
    reading_details_list.append(metadata)
    continue

  items = tracks_soup.find("div", {"class": "field__items"}).find_all("div", {"class": "field__item"}, recursive=False)

  # parse each track in the reading
  for track_soup in items:
    media_details_soup = track_soup.find("div", {"class": "media-details"})
    media_track_soup = media_details_soup.find("div", {"class": "field--name-field-media-track-title"})

    if media_track_soup == None:
      # Eavan Boland's 2003 is not properly broken down into tracks
      print("👀 Malformed metadata")
      continue

    track_metadata = {
      "track_name": media_track_soup.find("div", {"class": "field__item"}).text.strip(),
      "track_link": track_soup.find("a", {"class": "track-link"}).get("href")
    }

    # Sometimes there's no reader metadata on this level
    # Example: An Evening with Raquel Salas Rivera, Sin à Tes Souhaits (Frank Johnson), and Vanessa Angélica Villarreal - February 18, 2021
    reader_media_details_soup = media_details_soup.find("div", {"class": "field--name-field-media-reader"})
    if reader_media_details_soup:
      track_metadata['reader_name'] = reader_media_details_soup.find("a").text.strip()
      track_metadata['reader_link'] = reader_media_details_soup.find("a").get("href")
    else:
      print(f"👀 Missing reader for track {track_metadata['track_name']}") # could be interesting to investigate
      track_metadata['reader_name'] = ""
      track_metadata['reader_link'] = ""

    # MP4 and MP3 are both valid audio formats
    resolve_audio_file(track_soup, track_metadata)

    # sometimes captions need to be requested (unavailable in voca)
    if track_soup.find("div", {"class": "field--name-field-media-captions"}):
      track_metadata['captions_link'] = track_soup.find("div", {"class": "field--name-field-media-captions"}).find("a").get("href")
    else:
      print(f"👀 Missing captions for track {track_metadata['track_name']}") # could be interesting to investigate
      track_metadata['captions_link'] = ""


    # optional metadata - tags, description, and citation
    citation_soup = media_details_soup.find("div", {"class": "field--name-field-media-citation"})

    track_metadata['citation'] = get_field_item_if_exists(citation_soup)

    track_description_soup = media_details_soup.find("div", {"class": "field--name-field-media-description"})
    track_metadata['track_description'] = get_field_item_if_exists(track_description_soup)

    track_tags_soup = track_soup.find("div", {"class": "field--name-field-media-tag"})

    if track_tags_soup:
      track_metadata['tags'] = []
      for track_tag_soup in track_tags_soup.find_all("a"):
        track_metadata['tags'].append(track_tag_soup.text)
    else:
      track_metadata['tags'] = []

    tracks.append(track_metadata)
  metadata['tracks'] = tracks

  reading_details_list.append(metadata)

In [ ]:
# save reading_details_list to json
import json

with open('reading_details_list.json', 'w') as f:
    json.dump(reading_details_list, f)


In [ ]:
# for ease of querying with relational syntax, also create one table for reading metadata
# and one table for poems from reading_details_list
# they are linked via (reading_title, reading_link)

with open('reading_details_list.json') as f:
  reading_details_list = json.load(f)

track_metadata_only = []
for item in reading_details_list:
  print(item)
  common_keys = {
    "reading_title": item['reading_title'],
    "reading_link": item['reading_url']
  }
  for track in item['tracks']:
    track_metadata_only.append(track | common_keys)

track_metadata_df = pd.DataFrame(track_metadata_only)
track_metadata_df.to_csv('track_metadata_only.csv', index=False)

reading_metadata_only = reading_details_list.copy()
for item in reading_metadata_only:
  del item['tracks']

reading_metadata_df = pd.DataFrame(reading_metadata_only)
reading_metadata_df.to_csv('reading_metadata_only.csv', index=False)

